# Mapa semántico de tesis del CIDE

Este cuaderno construye un mapa semántico de las tesis de Licenciatura en Economía del CIDE usando embeddings multilingües, UMAP, clustering, etiquetas automáticas de temas, evolución temporal y cruces con asesores homologados.

La entrada canónica es `tesis_lic_economia_cide.parquet`. No hace scraping: si quieres actualizar datos, corre primero `make scrape`.


## Dependencias y configuración

El modelo por defecto es `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`: es más pesado que MiniLM, pero en el benchmark local separó mejor los temas y mezcló mejor español/inglés. Puedes cambiar el modelo con `ST_MODEL_NAME`, el número de clusters con `ST_CLUSTER_K` y el batch size con `ST_BATCH_SIZE` sin editar la notebook.


In [ ]:
import os
import re
import json
import html
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import umap
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from IPython.display import display, HTML
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.manifold import trustworthiness
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

RANDOM_STATE = 420
DATA_PATH = Path('tesis_lic_economia_cide.parquet')
EMBEDDINGS_PATH = Path('embeddings_tesis.parquet')
CLUSTERS_PATH = Path('clusters_tesis.parquet')
CLUSTER_SUMMARY_PATH = Path('clusters_resumen.parquet')
CLUSTER_DIAGNOSTICS_PATH = Path('cluster_diagnostics.parquet')
CLUSTER_YEAR_PATH = Path('cluster_anio.parquet')
CLUSTER_LANGUAGE_PATH = Path('cluster_idioma.parquet')
CLUSTER_LIFECYCLE_PATH = Path('cluster_lifecycle.parquet')
ADVISOR_CLUSTER_PATH = Path('asesor_cluster_resumen.parquet')
ADVISOR_SUMMARY_PATH = Path('asesor_resumen.parquet')
MODEL_BENCHMARK_PATH = Path('model_benchmark.parquet')
DASHBOARD_PATH = Path('semantic_dashboard.html')

MODEL_NAME = os.environ.get('ST_MODEL_NAME', 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
CLUSTER_K = int(os.environ.get('ST_CLUSTER_K', '11'))
DEVICE = os.environ.get('ST_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = int(os.environ.get('ST_BATCH_SIZE', '8' if 'mpnet' in MODEL_NAME.lower() else '16'))

pio.templates.default = 'plotly_white'
PALETTE = [
    '#2563eb', '#dc2626', '#059669', '#7c3aed', '#d97706', '#0891b2', '#be123c', '#4d7c0f',
    '#9333ea', '#0f766e', '#b45309', '#475569', '#db2777', '#64748b', '#ea580c', '#16a34a',
    '#1d4ed8', '#a16207', '#0369a1', '#c026d3', '#52525b', '#15803d'
]
LANGUAGE_COLOR_MAP = {'spa': '#2563eb', 'eng': '#f59e0b', 'desconocido': '#94a3b8'}

print(f'Modelo de embeddings: {MODEL_NAME}')
print(f'Dispositivo: {DEVICE}; batch_size={BATCH_SIZE}')
print(f'Clusters configurados: k={CLUSTER_K}')


## Cargar y preparar los datos

Se combinan título y resumen. La columna `asesor_unificado` ya viene homologada desde el pipeline de scraping.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'No encontré {DATA_PATH}. Corre primero ScrapingTesisLicEcoCIDE.qmd o make scrape para generar el Parquet canónico.'
    )

df_raw = pd.read_parquet(DATA_PATH)

required_cols = {'anio_pub', 'autorx', 'titulo', 'resumen', 'asesor_unificado', 'url'}
missing_cols = required_cols.difference(df_raw.columns)
if missing_cols:
    raise ValueError(f'Faltan columnas requeridas en {DATA_PATH}: {sorted(missing_cols)}')

df_raw['anio_pub'] = pd.to_numeric(df_raw['anio_pub'], errors='coerce').astype('Int64')

text_cols = ['titulo', 'resumen']
df = df_raw.dropna(subset=text_cols).copy()
df['texto'] = (df['titulo'].fillna('') + '. ' + df['resumen'].fillna('')).str.replace(r'\s+', ' ', regex=True).str.strip()
df = df[df['texto'].str.len() > 20].reset_index(drop=True)

df['asesor_unificado'] = df['asesor_unificado'].fillna('Asesor desconocido').replace('', 'Asesor desconocido')
if 'idioma' not in df.columns:
    df['idioma'] = 'desconocido'
else:
    df['idioma'] = df['idioma'].fillna('desconocido').replace('', 'desconocido')

df['periodo'] = pd.cut(
    df['anio_pub'].astype('float'),
    bins=[1979, 1989, 1999, 2009, 2019, 2030],
    labels=['1980s', '1990s', '2000s', '2010s', '2020s'],
)
df['text_hash'] = df['texto'].map(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest())

print(f'Tesis disponibles para analizar: {len(df)}')
print(f'Años: {df["anio_pub"].min()}-{df["anio_pub"].max()}')
print(f'Asesores homologados distintos: {df["asesor_unificado"].nunique()}')
print(f'Idiomas detectados: {df["idioma"].value_counts().to_dict()}')
display(df[['anio_pub', 'titulo', 'asesor_unificado', 'idioma', 'periodo', 'url']].head(3))


## Benchmark de modelos

El benchmark compara modelos multilingües sin volver a scrapear. La selección por defecto privilegia más estructura semántica y menos dependencia del idioma, aunque deja `ST_CLUSTER_K` como control explícito.


In [ ]:
if MODEL_BENCHMARK_PATH.exists():
    model_benchmark = pd.read_parquet(MODEL_BENCHMARK_PATH)
    benchmark_cols = [
        'model_name', 'embedding_dim', 'seconds', 'k', 'silhouette_cosine', 'davies_bouldin',
        'calinski_harabasz', 'language_nmi', 'min_cluster_size', 'max_cluster_size', 'umap_trustworthiness'
    ]
    benchmark_cols = [c for c in benchmark_cols if c in model_benchmark.columns]
    benchmark_view = (
        model_benchmark[benchmark_cols]
        .sort_values(['silhouette_cosine', 'umap_trustworthiness'], ascending=[False, False])
        .reset_index(drop=True)
    )
    display(benchmark_view.head(15).style.format({
        'seconds': '{:.1f}',
        'silhouette_cosine': '{:.4f}',
        'davies_bouldin': '{:.3f}',
        'calinski_harabasz': '{:.2f}',
        'language_nmi': '{:.4f}',
        'umap_trustworthiness': '{:.4f}',
    }))
else:
    model_benchmark = pd.DataFrame()
    benchmark_view = pd.DataFrame()
    print(f'No encontré {MODEL_BENCHMARK_PATH}. Ejecuta el benchmark de modelos si quieres comparar candidatos.')


## Generar o cargar embeddings multilingües

Los embeddings se guardan en Parquet para no recalcularlos si el texto y el modelo no cambiaron. Esto permite iterar sobre UMAP, clustering y visualizaciones sin volver a descargar ni ejecutar el transformer.


In [ ]:
def embedding_columns(frame):
    return [c for c in frame.columns if c.startswith('embedding_')]

cache_ok = False
if EMBEDDINGS_PATH.exists():
    cached = pd.read_parquet(EMBEDDINGS_PATH)
    emb_cols = embedding_columns(cached)
    cache_ok = (
        len(cached) == len(df)
        and emb_cols
        and set(['url', 'text_hash', 'model_name']).issubset(cached.columns)
        and cached['url'].tolist() == df['url'].tolist()
        and cached['text_hash'].tolist() == df['text_hash'].tolist()
        and cached['model_name'].nunique() == 1
        and cached['model_name'].iloc[0] == MODEL_NAME
    )

if cache_ok:
    embeddings = cached[emb_cols].to_numpy(dtype=np.float32)
    print(f'Embeddings cargados desde cache: {EMBEDDINGS_PATH} {embeddings.shape}')
else:
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    encode_texts = df['texto'].tolist()
    if 'multilingual-e5' in MODEL_NAME.lower() or MODEL_NAME.lower().startswith('intfloat/e5'):
        encode_texts = [f'passage: {text}' for text in encode_texts]

    embeddings = model.encode(
        encode_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    emb_cols = [f'embedding_{i:03d}' for i in range(embeddings.shape[1])]
    embeddings_export = pd.DataFrame(embeddings, columns=emb_cols)
    embeddings_export.insert(0, 'model_name', MODEL_NAME)
    embeddings_export.insert(0, 'text_hash', df['text_hash'])
    embeddings_export.insert(0, 'url', df['url'])
    embeddings_export.to_parquet(EMBEDDINGS_PATH, index=False)
    print(f'Embeddings generados y guardados: {EMBEDDINGS_PATH.resolve()} {embeddings.shape}')

if not np.all(np.isfinite(embeddings)):
    raise ValueError('Los embeddings contienen valores no finitos.')


## Reducir dimensionalidad con UMAP

UMAP se usa como layout visual. El clustering principal se calcula sobre embeddings originales para evitar que la geometría 2D determine artificialmente los grupos.


In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=18,
    min_dist=0.025,
    metric='cosine',
    random_state=RANDOM_STATE,
)

umap_2d = umap_model.fit_transform(embeddings)
umap_trust = trustworthiness(embeddings, umap_2d, n_neighbors=18, metric='cosine')

df_layout = df.copy()
df_layout['umap_x'] = umap_2d[:, 0]
df_layout['umap_y'] = umap_2d[:, 1]
print(f'Trustworthiness UMAP: {umap_trust:.4f}')
display(df_layout[['titulo', 'anio_pub', 'idioma', 'umap_x', 'umap_y']].head(3))


## Diagnóstico de número de clusters

Se comparan K-Means sobre embeddings originales y sobre UMAP 2D para varios valores de `k`. La selección automática usa silhouette sobre embeddings, que es más cercana al espacio semántico real.


In [ ]:
language_labels = df_layout['idioma'].fillna('desconocido').astype(str)

def evaluate_kmeans(features, feature_space, k_values, metric_for_silhouette):
    rows = []
    for k in k_values:
        if k >= len(features):
            continue
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30)
        labels = km.fit_predict(features)
        silhouette_value = silhouette_score(features, labels, metric=metric_for_silhouette)
        rows.append({
            'model_name': MODEL_NAME,
            'feature_space': feature_space,
            'k': k,
            'silhouette': silhouette_value,
            'silhouette_metric': metric_for_silhouette,
            'davies_bouldin': davies_bouldin_score(features, labels),
            'calinski_harabasz': calinski_harabasz_score(features, labels),
            'inertia': km.inertia_,
            'language_nmi': normalized_mutual_info_score(language_labels, labels) if language_labels.nunique() > 1 else np.nan,
            'min_cluster_size': int(pd.Series(labels).value_counts().min()),
            'max_cluster_size': int(pd.Series(labels).value_counts().max()),
            'umap_trustworthiness': umap_trust,
        })
    return pd.DataFrame(rows)

k_values = list(range(6, min(18, len(df_layout))))
diagnostics = pd.concat(
    [
        evaluate_kmeans(embeddings, 'embeddings', k_values, 'cosine'),
        evaluate_kmeans(umap_2d, 'umap_2d', k_values, 'euclidean'),
    ],
    ignore_index=True,
)

embedding_diag = diagnostics[diagnostics['feature_space'] == 'embeddings'].copy()
best_row = embedding_diag.sort_values(['silhouette', 'davies_bouldin'], ascending=[False, True]).iloc[0]
best_silhouette_k = int(best_row['k'])
selected_k = int(min(max(CLUSTER_K, 2), len(df_layout) - 1))

diagnostics['selected'] = diagnostics['feature_space'].eq('embeddings') & diagnostics['k'].eq(selected_k)
diagnostics['best_silhouette'] = diagnostics['feature_space'].eq('embeddings') & diagnostics['k'].eq(best_silhouette_k)
diagnostics['selection_reason'] = diagnostics['k'].map(lambda k: 'k configurado por ST_CLUSTER_K' if k == selected_k else '')
diagnostics['generated_at'] = datetime.now(timezone.utc).isoformat()
diagnostics.to_parquet(CLUSTER_DIAGNOSTICS_PATH, index=False)

print(f'k seleccionado: {selected_k} (configurado)')
print(f'k con mejor silhouette en embeddings: {best_silhouette_k}')
display(diagnostics.sort_values(['feature_space', 'k']))

fig_diag = px.line(
    diagnostics,
    x='k',
    y='silhouette',
    color='feature_space',
    markers=True,
    title='Diagnóstico de clusters por silhouette',
    width=940,
    height=430,
    labels={'silhouette': 'Silhouette', 'feature_space': 'Espacio'},
)
fig_diag.add_vline(x=selected_k, line_dash='dash', line_color='#111827', annotation_text=f'k usado: {selected_k}')
fig_diag.add_vline(x=best_silhouette_k, line_dash='dot', line_color='#dc2626', annotation_text=f'mejor silhouette: {best_silhouette_k}')
fig_diag.update_layout(legend_title_text='Espacio', margin=dict(l=40, r=30, t=70, b=40))
fig_diag.show()


## Clustering semántico sobre embeddings

El cluster final se calcula en el espacio de embeddings multilingües. UMAP queda como visualización.


In [ ]:
kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=60)
cluster_labels = kmeans.fit_predict(embeddings)
cluster_algo = 'kmeans (embeddings)'

df_layout['cluster_id'] = cluster_labels
df_layout['cluster_label'] = df_layout['cluster_id'].map(lambda x: f'Cluster {x:02d}')
cluster_order = [f'Cluster {cid:02d}' for cid in sorted(df_layout['cluster_id'].unique())]
cluster_color_map = {label: PALETTE[i % len(PALETTE)] for i, label in enumerate(cluster_order)}

print(f'Algoritmo de clustering activo: {cluster_algo} (k={selected_k})')

cluster_summary_base = (
    df_layout.groupby(['cluster_label', 'cluster_id'])
    .agg(
        conteo=('titulo', 'count'),
        anio_promedio=('anio_pub', 'mean'),
        anio_min=('anio_pub', 'min'),
        anio_max=('anio_pub', 'max'),
        idiomas=('idioma', lambda s: ', '.join(s.value_counts().head(3).index)),
    )
    .reset_index()
    .sort_values('conteo', ascending=False)
)

display(cluster_summary_base)


## Etiquetar clusters con keywords y tesis representativas

Se usa TF-IDF por cluster para términos característicos y similitud al centroide para tesis representativas.


In [ ]:
SPANISH_STOPWORDS = {
    'a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra', 'cual', 'cuando',
    'de', 'del', 'desde', 'donde', 'dos', 'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre',
    'era', 'eran', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estaba', 'estado', 'estados',
    'estan', 'estar', 'estas', 'este', 'esto', 'estos', 'fue', 'fueron', 'ha', 'hacia', 'han', 'hasta',
    'hay', 'la', 'las', 'le', 'les', 'lo', 'los', 'mas', 'más', 'me', 'mediante', 'mi', 'mientras',
    'muy', 'no', 'nos', 'o', 'para', 'pero', 'por', 'porque', 'que', 'se', 'sin', 'sobre', 'son',
    'su', 'sus', 'tambien', 'también', 'te', 'tiene', 'tienen', 'un', 'una', 'uno', 'unos', 'y', 'ya',
    'méxico', 'mexico', 'cide', 'tesis', 'licenciatura', 'economía', 'economia', 'análisis', 'analisis',
    'modelo', 'modelos', 'caso', 'estudio', 'evidencia', 'efecto', 'efectos', 'datos', 'resultados',
    'trabajo', 'seccion', 'sección', 'capitulo', 'capítulo', 'presente', 'objetivo', 'objetivos',
    'literatura', 'investigacion', 'investigación', 'variables', 'variable', 'muestra', 'metodologia',
    'metodología', 'using', 'based', 'paper', 'thesis', 'chapter', 'section'
}
STOPWORDS = sorted(set(ENGLISH_STOP_WORDS).union(SPANISH_STOPWORDS))

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=STOPWORDS,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
    max_features=6500,
)

tfidf = vectorizer.fit_transform(df_layout['texto'])
terms = np.array(vectorizer.get_feature_names_out())

keywords_by_cluster = {}
for cid in sorted(df_layout['cluster_id'].unique()):
    mask = df_layout['cluster_id'].to_numpy() == cid
    scores = np.asarray(tfidf[mask].mean(axis=0)).ravel()
    top_idx = scores.argsort()[::-1][:12]
    keywords_by_cluster[cid] = ', '.join(terms[top_idx])

embeddings_norm = normalize(embeddings)
centroids_norm = normalize(kmeans.cluster_centers_)
representatives = {}
for cid in sorted(df_layout['cluster_id'].unique()):
    mask_idx = np.where(df_layout['cluster_id'].to_numpy() == cid)[0]
    sims = cosine_similarity(embeddings_norm[mask_idx], centroids_norm[[cid]]).ravel()
    top_local = mask_idx[np.argsort(sims)[::-1][:3]]
    representatives[cid] = ' | '.join(df_layout.loc[top_local, 'titulo'].tolist())

top_advisors = (
    df_layout.groupby(['cluster_id', 'asesor_unificado'])
    .size()
    .reset_index(name='n')
    .sort_values(['cluster_id', 'n'], ascending=[True, False])
    .groupby('cluster_id')
    .head(5)
    .groupby('cluster_id')
    .apply(lambda g: ', '.join(f'{r.asesor_unificado} ({r.n})' for r in g.itertuples()), include_groups=False)
    .to_dict()
)

CURATED_CLUSTER_THEMES = {
    0: 'Educación y desempeño escolar',
    1: 'Crédito y sistema financiero',
    2: 'Industria, productividad e innovación',
    3: 'Macroeconomía y política económica',
    4: 'Género, familia y mercado laboral',
    5: 'Desarrollo, pobreza y migración',
    6: 'Crimen, seguridad y ciudad',
    7: 'Organización industrial y teoría de juegos',
    8: 'Finanzas, riesgo y mercados',
    9: 'Agricultura, medio ambiente y clima',
    10: 'Salud, bienestar y protección social',
}

def default_cluster_theme(cid):
    pieces = [piece.strip() for piece in keywords_by_cluster.get(cid, '').split(',') if piece.strip()]
    return ' / '.join(pieces[:3]).title() if pieces else f'Tema {cid:02d}'

cluster_theme_by_id = {
    cid: CURATED_CLUSTER_THEMES.get(cid, default_cluster_theme(cid))
    for cid in sorted(df_layout['cluster_id'].unique())
}
df_layout['cluster_theme'] = df_layout['cluster_id'].map(cluster_theme_by_id)
df_layout['cluster_display_label'] = df_layout.apply(
    lambda r: f"{r['cluster_label']}: {r['cluster_theme']}",
    axis=1,
)
cluster_theme_order = [cluster_theme_by_id[cid] for cid in sorted(df_layout['cluster_id'].unique())]
cluster_display_order = [f"Cluster {cid:02d}: {cluster_theme_by_id[cid]}" for cid in sorted(df_layout['cluster_id'].unique())]
cluster_theme_color_map = {
    cluster_theme_by_id[cid]: cluster_color_map[f'Cluster {cid:02d}']
    for cid in sorted(df_layout['cluster_id'].unique())
}
cluster_display_color_map = {
    f"Cluster {cid:02d}: {cluster_theme_by_id[cid]}": cluster_color_map[f'Cluster {cid:02d}']
    for cid in sorted(df_layout['cluster_id'].unique())
}
cluster_label_to_theme = {f'Cluster {cid:02d}': theme for cid, theme in cluster_theme_by_id.items()}
cluster_label_to_display = {label: f'{label}: {theme}' for label, theme in cluster_label_to_theme.items()}

cluster_overview = cluster_summary_base.copy()
cluster_overview['cluster_theme'] = cluster_overview['cluster_id'].map(cluster_theme_by_id)
cluster_overview['cluster_display_label'] = cluster_overview.apply(
    lambda r: f"{r['cluster_label']}: {r['cluster_theme']}",
    axis=1,
)
cluster_overview['keywords'] = cluster_overview['cluster_id'].map(keywords_by_cluster)
cluster_overview['tesis_representativas'] = cluster_overview['cluster_id'].map(representatives)
cluster_overview['asesores_top'] = cluster_overview['cluster_id'].map(top_advisors)
cluster_overview['cluster_algo'] = cluster_algo
cluster_overview['embedding_model'] = MODEL_NAME
cluster_overview['umap_trustworthiness'] = umap_trust
cluster_overview = cluster_overview.sort_values('conteo', ascending=False)

display(cluster_overview[['cluster_label', 'cluster_theme', 'conteo', 'anio_promedio', 'idiomas', 'keywords', 'tesis_representativas']])


## Mapa semántico interactivo


In [ ]:
hover_cols = {
    'cluster_label': True,
    'cluster_theme': False,
    'titulo': True,
    'autorx': True,
    'anio_pub': True,
    'asesor_unificado': True,
    'idioma': True,
    'periodo': True,
    'url': True,
    'umap_x': False,
    'umap_y': False,
}

fig_map = px.scatter(
    df_layout,
    x='umap_x',
    y='umap_y',
    color='cluster_theme',
    category_orders={'cluster_theme': cluster_theme_order},
    color_discrete_map=cluster_theme_color_map,
    hover_name='titulo',
    hover_data=hover_cols,
    title='Mapa semántico acumulado de tesis de Lic. en Economía (CIDE)',
    width=1100,
    height=740,
)
fig_map.update_traces(marker=dict(size=8.5, opacity=0.84, line=dict(width=0.45, color='white')))
fig_map.update_layout(legend_title_text='Tema', margin=dict(l=30, r=30, t=70, b=30), plot_bgcolor='white')
fig_map.update_xaxes(showticklabels=False, title='', showgrid=False, zeroline=False)
fig_map.update_yaxes(showticklabels=False, title='', showgrid=False, zeroline=False)
fig_map.show()


## Máquina temporal del mapa semántico

Esta vista usa una geometría UMAP fija y anima la aparición de tesis por año. Los puntos grises son tesis publicadas antes del año activo; los puntos de color son las tesis nuevas del año, codificadas por tema. Así la película muestra cómo se llena el espacio semántico sin hacer que los puntos “se muevan” artificialmente.


In [ ]:
year_values = df_layout['anio_pub'].dropna().astype(int)
timeline_years = sorted(year_values.unique().tolist())

if not timeline_years:
    fig_time_machine = None
    fig_period_facets = None
    print('No hay años de publicación suficientes para construir la vista temporal.')
else:
    x_range = [df_layout['umap_x'].min() - 0.7, df_layout['umap_x'].max() + 0.7]
    y_range = [df_layout['umap_y'].min() - 0.7, df_layout['umap_y'].max() + 0.7]

    def temporal_customdata(frame):
        if frame.empty:
            return np.empty((0, 6))
        return frame[['titulo', 'autorx', 'asesor_unificado', 'anio_pub', 'idioma', 'url']].fillna('').to_numpy()

    def movie_title(year):
        accumulated = int(df_layout['anio_pub'].astype('Int64').le(year).sum())
        previous = int(df_layout['anio_pub'].astype('Int64').lt(year).sum())
        new = int(df_layout['anio_pub'].astype('Int64').eq(year).sum())
        active_themes = int(df_layout.loc[df_layout['anio_pub'].astype('Int64').eq(year), 'cluster_theme'].nunique())
        return f'Mapa semántico acumulado hasta {year} · {previous} previas · {new} nuevas · {active_themes} temas activos'

    def cumulative_traces_for_year(year, showlegend=False):
        history = df_layout[df_layout['anio_pub'].astype('Int64').lt(year)]
        current = df_layout[df_layout['anio_pub'].astype('Int64').eq(year)]
        traces = [
            go.Scatter(
                x=history['umap_x'],
                y=history['umap_y'],
                mode='markers',
                name='Tesis previas',
                marker=dict(size=5.6, color='rgba(100,116,139,0.28)', line=dict(width=0)),
                customdata=temporal_customdata(history),
                hovertemplate=(
                    '<b>%{customdata[0]}</b><br>'
                    'Autor/a: %{customdata[1]}<br>'
                    'Asesor/a: %{customdata[2]}<br>'
                    'Año tesis: %{customdata[3]}<br>'
                    'Idioma: %{customdata[4]}<br>'
                    '<extra>Tesis previa</extra>'
                ),
                showlegend=showlegend,
            )
        ]

        for theme in cluster_theme_order:
            subset = current[current['cluster_theme'].eq(theme)]
            traces.append(
                go.Scatter(
                    x=subset['umap_x'],
                    y=subset['umap_y'],
                    mode='markers',
                    name=theme,
                    legendgroup=theme,
                    showlegend=showlegend,
                    marker=dict(
                        size=12.5,
                        color=cluster_theme_color_map.get(theme, '#2563eb'),
                        opacity=0.96,
                        line=dict(width=0.85, color='white'),
                    ),
                    customdata=temporal_customdata(subset),
                    hovertemplate=(
                        '<b>%{customdata[0]}</b><br>'
                        'Autor/a: %{customdata[1]}<br>'
                        'Asesor/a: %{customdata[2]}<br>'
                        'Año tesis: %{customdata[3]}<br>'
                        'Idioma: %{customdata[4]}<br>'
                        '<extra>Tesis nueva</extra>'
                    ),
                )
            )
        return traces

    first_year = timeline_years[0]
    frame_trace_ids = list(range(1 + len(cluster_theme_order)))
    frames = [
        go.Frame(
            name=str(year),
            data=cumulative_traces_for_year(year, showlegend=False),
            traces=frame_trace_ids,
            layout=go.Layout(title_text=movie_title(year)),
        )
        for year in timeline_years
    ]

    fig_time_machine = go.Figure(data=cumulative_traces_for_year(first_year, showlegend=True), frames=frames)
    slider_steps = [
        dict(
            method='animate',
            label=str(year),
            args=[
                [str(year)],
                dict(mode='immediate', frame=dict(duration=560, redraw=True), transition=dict(duration=260, easing='cubic-in-out')),
            ],
        )
        for year in timeline_years
    ]
    fig_time_machine.update_layout(
        title=movie_title(first_year),
        width=1120,
        height=770,
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend_title_text='Tema nuevo del año',
        margin=dict(l=25, r=25, t=78, b=76),
        xaxis=dict(range=x_range, visible=False),
        yaxis=dict(range=y_range, visible=False, scaleanchor='x', scaleratio=1),
        updatemenus=[
            dict(
                type='buttons',
                direction='left',
                x=0,
                y=-0.055,
                xanchor='left',
                yanchor='top',
                buttons=[
                    dict(
                        label='Play',
                        method='animate',
                        args=[None, dict(frame=dict(duration=820, redraw=True), fromcurrent=True, transition=dict(duration=320, easing='cubic-in-out'))],
                    ),
                    dict(
                        label='Pause',
                        method='animate',
                        args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate', transition=dict(duration=0))],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                currentvalue=dict(prefix='Año ', font=dict(size=13)),
                pad=dict(t=44, b=5),
                steps=slider_steps,
            )
        ],
    )
    fig_time_machine.show()

    period_order = [p for p in ['1990s', '2000s', '2010s', '2020s'] if p in set(df_layout['periodo'].dropna())]
    fig_period_facets = px.scatter(
        df_layout.dropna(subset=['periodo']),
        x='umap_x',
        y='umap_y',
        color='cluster_theme',
        facet_col='periodo',
        facet_col_wrap=2,
        category_orders={'cluster_theme': cluster_theme_order, 'periodo': period_order},
        color_discrete_map=cluster_theme_color_map,
        hover_name='titulo',
        hover_data=hover_cols,
        title='Mapa semántico por periodo de publicación',
        width=1120,
        height=720,
    )
    fig_period_facets.update_traces(marker=dict(size=7.2, opacity=0.76, line=dict(width=0.35, color='white')))
    fig_period_facets.for_each_annotation(lambda a: a.update(text=a.text.replace('periodo=', '')))
    fig_period_facets.update_layout(legend_title_text='Tema', margin=dict(l=20, r=20, t=72, b=30), plot_bgcolor='white')
    fig_period_facets.update_xaxes(showticklabels=False, title='', showgrid=False, zeroline=False, matches=None)
    fig_period_facets.update_yaxes(showticklabels=False, title='', showgrid=False, zeroline=False, matches=None)
    fig_period_facets.show()


## Evolución temporal de temas

Se genera una grilla completa año × tema para ver tanto presencia como ausencia. El heatmap ayuda a detectar picos puntuales; el streamgraph muestra cómo cambia la mezcla temática de cada cohorte anual.

In [ ]:

cluster_year_observed = (
    df_layout.dropna(subset=['anio_pub'])
    .assign(anio_pub=lambda d: d['anio_pub'].astype(int))
    .groupby(['anio_pub', 'cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label'])
    .size()
    .reset_index(name='conteo')
)

all_years = list(range(int(df_layout['anio_pub'].min()), int(df_layout['anio_pub'].max()) + 1))
cluster_meta = (
    df_layout[['cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label']]
    .drop_duplicates()
    .sort_values('cluster_id')
)
cluster_year_grid = (
    pd.MultiIndex.from_product([all_years, cluster_order], names=['anio_pub', 'cluster_label'])
    .to_frame(index=False)
    .merge(cluster_meta, on='cluster_label', how='left')
    .merge(
        cluster_year_observed[['anio_pub', 'cluster_label', 'conteo']],
        on=['anio_pub', 'cluster_label'],
        how='left',
    )
)
cluster_year_grid['conteo'] = cluster_year_grid['conteo'].fillna(0).astype(int)
year_totals = df_layout.dropna(subset=['anio_pub']).assign(anio_pub=lambda d: d['anio_pub'].astype(int)).groupby('anio_pub').size()
cluster_year_grid['total_anio'] = cluster_year_grid['anio_pub'].map(year_totals).fillna(0).astype(int)
cluster_year_grid['participacion_anio'] = np.where(
    cluster_year_grid['total_anio'] > 0,
    cluster_year_grid['conteo'] / cluster_year_grid['total_anio'],
    0.0,
)
cluster_year = cluster_year_grid.copy()
cluster_year.to_parquet(CLUSTER_YEAR_PATH, index=False)

heatmap_data = cluster_year.pivot_table(
    index='cluster_theme',
    columns='anio_pub',
    values='participacion_anio',
    fill_value=0,
).reindex(cluster_theme_order)

fig_year_heatmap = px.imshow(
    heatmap_data,
    aspect='auto',
    color_continuous_scale='Blues',
    title='Participación de cada tema por año',
    labels=dict(x='Año', y='Tema', color='Participación'),
    width=1060,
    height=620,
)
fig_year_heatmap.update_layout(margin=dict(l=210, r=30, t=70, b=50))
fig_year_heatmap.show()

fig_theme_stream = go.Figure()
for cluster_name in cluster_order:
    subset = cluster_year[cluster_year['cluster_label'].eq(cluster_name)].sort_values('anio_pub')
    theme = cluster_label_to_theme.get(cluster_name, cluster_name)
    fig_theme_stream.add_trace(
        go.Scatter(
            x=subset['anio_pub'],
            y=subset['participacion_anio'] * 100,
            mode='lines',
            stackgroup='one',
            name=theme,
            line=dict(width=0.5, color=cluster_color_map.get(cluster_name, '#2563eb')),
            hovertemplate=f'{theme}<br>Año: %{{x}}<br>Participación: %{{y:.1f}}%<extra></extra>',
        )
    )
fig_theme_stream.update_layout(
    title='Composición temática anual de las tesis',
    width=1060,
    height=600,
    yaxis=dict(title='Participación anual (%)', range=[0, 100], ticksuffix='%'),
    xaxis=dict(title='Año', dtick=2),
    legend_title_text='Tema',
    margin=dict(l=45, r=25, t=70, b=45),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig_theme_stream.show()


def race_trace_for_year(year):
    subset = (
        cluster_year[cluster_year['anio_pub'].eq(year)]
        .assign(participacion_pct=lambda d: d['participacion_anio'] * 100)
        .sort_values('participacion_pct', ascending=True)
    )
    labels = subset['cluster_theme'].tolist()
    colors = subset['cluster_label'].map(cluster_color_map).tolist()
    return go.Bar(
        x=subset['participacion_pct'],
        y=labels,
        orientation='h',
        marker=dict(color=colors, line=dict(color='white', width=0.6)),
        text=subset['conteo'].map(lambda x: f'{int(x)}'),
        textposition='outside',
        cliponaxis=False,
        hovertemplate='%{y}<br>Participación: %{x:.1f}%<br>Tesis: %{text}<extra></extra>',
    )

def race_title(year):
    total = int(cluster_year.loc[cluster_year['anio_pub'].eq(year), 'total_anio'].max())
    return f'Temas dominantes en {year} · {total} tesis'

race_years = [year for year in all_years if int(year_totals.get(year, 0)) > 0]
if race_years:
    first_race_year = race_years[0]
    fig_theme_race = go.Figure(data=[race_trace_for_year(first_race_year)])
    fig_theme_race.frames = [
        go.Frame(name=str(year), data=[race_trace_for_year(year)], layout=go.Layout(title_text=race_title(year)))
        for year in race_years
    ]
    race_steps = [
        dict(
            method='animate',
            label=str(year),
            args=[[str(year)], dict(mode='immediate', frame=dict(duration=520, redraw=True), transition=dict(duration=180))],
        )
        for year in race_years
    ]
    fig_theme_race.update_layout(
        title=race_title(first_race_year),
        width=1060,
        height=600,
        xaxis=dict(title='Participación anual (%)', range=[0, max(55, cluster_year['participacion_anio'].max() * 115)], ticksuffix='%'),
        yaxis=dict(title='', automargin=True),
        margin=dict(l=210, r=45, t=70, b=80),
        plot_bgcolor='white',
        paper_bgcolor='white',
        showlegend=False,
        updatemenus=[
            dict(
                type='buttons',
                direction='left',
                x=0,
                y=-0.12,
                xanchor='left',
                yanchor='top',
                buttons=[
                    dict(label='▶', method='animate', args=[None, dict(frame=dict(duration=760, redraw=True), fromcurrent=True, transition=dict(duration=220))]),
                    dict(label='II', method='animate', args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate', transition=dict(duration=0))]),
                ],
            )
        ],
        sliders=[dict(active=0, currentvalue=dict(prefix='Año ', font=dict(size=13)), pad=dict(t=48, b=5), steps=race_steps)],
    )
    fig_theme_race.show()
else:
    fig_theme_race = None

period_mix = (
    df_layout.dropna(subset=['periodo'])
    .groupby(['periodo', 'cluster_label'])
    .size()
    .reset_index(name='conteo_periodo')
)
period_mix['total_periodo'] = period_mix.groupby('periodo')['conteo_periodo'].transform('sum')
period_mix['participacion_periodo'] = period_mix['conteo_periodo'] / period_mix['total_periodo']
period_share = period_mix.pivot_table(
    index='cluster_label',
    columns='periodo',
    values='participacion_periodo',
    fill_value=0,
)

first_year = (
    cluster_year_observed.groupby('cluster_label')['anio_pub']
    .min()
    .rename('anio_inicio')
)
peak_rows = (
    cluster_year.sort_values(['cluster_label', 'conteo', 'participacion_anio', 'anio_pub'], ascending=[True, False, False, True])
    .groupby('cluster_label')
    .head(1)
    [['cluster_label', 'anio_pub', 'conteo', 'participacion_anio']]
    .rename(columns={'anio_pub': 'anio_pico', 'conteo': 'conteo_pico', 'participacion_anio': 'participacion_pico'})
)
cluster_lifecycle = (
    cluster_overview[['cluster_label', 'cluster_id', 'cluster_theme', 'conteo', 'keywords']]
    .merge(first_year, on='cluster_label', how='left')
    .merge(peak_rows, on='cluster_label', how='left')
    .sort_values('cluster_id')
)
cluster_lifecycle['participacion_pico'] = (cluster_lifecycle['participacion_pico'] * 100).round(1)
for period in ['2000s', '2010s', '2020s']:
    cluster_lifecycle[f'participacion_{period}'] = (
        cluster_lifecycle['cluster_label'].map(period_share.get(period, pd.Series(dtype=float))).fillna(0) * 100
    ).round(1)
cluster_lifecycle['cambio_2020s_vs_2010s_pp'] = (
    cluster_lifecycle.get('participacion_2020s', 0) - cluster_lifecycle.get('participacion_2010s', 0)
).round(1)
cluster_lifecycle.to_parquet(CLUSTER_LIFECYCLE_PATH, index=False)

display(cluster_lifecycle[['cluster_label', 'cluster_theme', 'conteo', 'anio_inicio', 'anio_pico', 'conteo_pico', 'participacion_pico', 'cambio_2020s_vs_2010s_pp']])


## Distribución de idioma por cluster

Esta tabla verifica si el modelo multilingüe está separando tesis por tema en vez de aislarlas por idioma. Si las tesis en inglés aparecen repartidas entre varios clusters, el espacio semántico está funcionando mejor para análisis bilingüe.


In [ ]:
if 'idioma' in df_layout.columns:
    cluster_language = (
        df_layout.assign(idioma=df_layout['idioma'].fillna('desconocido'))
        .groupby(['cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label', 'idioma'])
        .size()
        .reset_index(name='conteo')
    )
    cluster_language['total_cluster'] = cluster_language.groupby('cluster_label')['conteo'].transform('sum')
    cluster_language['participacion_cluster'] = cluster_language['conteo'] / cluster_language['total_cluster']
    cluster_language.to_parquet(CLUSTER_LANGUAGE_PATH, index=False)

    display(cluster_language.sort_values(['cluster_id', 'idioma']))

    fig_language = px.bar(
        cluster_language,
        x='cluster_theme',
        y='conteo',
        color='idioma',
        category_orders={'cluster_theme': cluster_theme_order},
        color_discrete_map=LANGUAGE_COLOR_MAP,
        title='Distribución de idioma por tema',
        width=1040,
        height=520,
    )
    fig_language.update_layout(
        barmode='stack', xaxis_title='Tema', yaxis_title='Número de tesis',
        legend_title_text='Idioma', margin=dict(l=50, r=30, t=70, b=165)
    )
    fig_language.update_xaxes(tickangle=35)
    fig_language.show()
else:
    cluster_language = pd.DataFrame(columns=['cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label', 'idioma', 'conteo', 'total_cluster', 'participacion_cluster'])
    cluster_language.to_parquet(CLUSTER_LANGUAGE_PATH, index=False)
    fig_language = None
    print('No hay columna idioma para evaluar distribución por cluster.')


## Cruce asesores × temas

Ahora que los asesores están homologados, se puede medir concentración temática y diversidad por asesor.


In [ ]:
advisor_cluster = (
    df_layout.groupby(['asesor_unificado', 'cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label'])
    .agg(n_tesis=('titulo', 'count'), anio_min=('anio_pub', 'min'), anio_max=('anio_pub', 'max'))
    .reset_index()
)
advisor_cluster['total_asesor'] = advisor_cluster.groupby('asesor_unificado')['n_tesis'].transform('sum')
advisor_cluster['participacion_asesor'] = advisor_cluster['n_tesis'] / advisor_cluster['total_asesor']
advisor_cluster = advisor_cluster.sort_values(['total_asesor', 'asesor_unificado', 'n_tesis'], ascending=[False, True, False])
advisor_cluster.to_parquet(ADVISOR_CLUSTER_PATH, index=False)

advisor_summary = (
    advisor_cluster.sort_values(['asesor_unificado', 'n_tesis'], ascending=[True, False])
    .groupby('asesor_unificado')
    .agg(
        n_tesis=('n_tesis', 'sum'),
        n_clusters=('cluster_id', 'nunique'),
        cluster_principal=('cluster_theme', 'first'),
        cluster_principal_id=('cluster_label', 'first'),
        participacion_cluster_principal=('participacion_asesor', 'first'),
        anio_min=('anio_min', 'min'),
        anio_max=('anio_max', 'max'),
    )
    .reset_index()
    .sort_values('n_tesis', ascending=False)
)
advisor_summary.to_parquet(ADVISOR_SUMMARY_PATH, index=False)

display(advisor_summary.head(20))

fig_advisor = px.scatter(
    advisor_summary.head(35),
    x='n_tesis', y='n_clusters', size='n_tesis', color='participacion_cluster_principal',
    hover_name='asesor_unificado', hover_data=['cluster_principal', 'cluster_principal_id', 'anio_min', 'anio_max'],
    title='Volumen y diversidad temática por asesor/a', color_continuous_scale='Tealrose',
    width=960, height=560,
    labels={
        'n_tesis': 'Tesis asesoradas',
        'n_clusters': 'Temas distintos',
        'participacion_cluster_principal': 'Concentración en tema principal',
    },
)
fig_advisor.update_traces(marker=dict(line=dict(width=0.6, color='white'), opacity=0.86))
fig_advisor.update_layout(margin=dict(l=60, r=30, t=70, b=50))
fig_advisor.show()


## Top asesores por cluster


In [ ]:
top_advisor_figs = []
for cid in sorted(df_layout['cluster_id'].unique()):
    cluster_label = f'Cluster {cid:02d}'
    cluster_theme = cluster_theme_by_id.get(cid, cluster_label)
    subset = df_layout[df_layout['cluster_id'] == cid]
    top5 = (
        subset.groupby('asesor_unificado')
        .size()
        .reset_index(name='conteo')
        .sort_values('conteo', ascending=False)
        .head(5)
    )

    if top5.empty:
        continue

    fig_top_advisors = px.bar(
        top5, x='conteo', y='asesor_unificado', orientation='h',
        title=f'Top 5 asesores: {cluster_theme}', text='conteo',
        color_discrete_sequence=[cluster_color_map.get(cluster_label, '#2563eb')],
    )
    fig_top_advisors.update_traces(textposition='outside', marker_line_width=0)
    fig_top_advisors.update_layout(
        height=320, xaxis_title='Número de tesis', yaxis_title='Asesor/a',
        yaxis=dict(categoryorder='array', categoryarray=top5['asesor_unificado'].tolist()[::-1]),
        showlegend=False, margin=dict(t=60, b=40, l=40, r=20),
    )
    top_advisor_figs.append(fig_top_advisors)
    fig_top_advisors.show()


## Exportar resultados analíticos


In [ ]:
cols_export = [
    'cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label', 'titulo', 'resumen',
    'autorx', 'anio_pub', 'periodo', 'asesor_unificado', 'idioma', 'url',
]

clusters_export = (
    df_layout[cols_export + ['umap_x', 'umap_y']]
    .sort_values(['cluster_id', 'anio_pub', 'titulo'])
    .reset_index(drop=True)
)

clusters_export.to_parquet(CLUSTERS_PATH, index=False)
cluster_overview.to_parquet(CLUSTER_SUMMARY_PATH, index=False)

print(f'Archivo exportado: {CLUSTERS_PATH.resolve()}')
print(f'Resumen exportado: {CLUSTER_SUMMARY_PATH.resolve()}')
print(f'Diagnóstico exportado: {CLUSTER_DIAGNOSTICS_PATH.resolve()}')
print(f'Evolución temporal exportada: {CLUSTER_YEAR_PATH.resolve()}')
print(f'Distribución por idioma exportada: {CLUSTER_LANGUAGE_PATH.resolve()}')
print(f'Cruce asesor-cluster exportado: {ADVISOR_CLUSTER_PATH.resolve()}')
print(f'Resumen de asesores exportado: {ADVISOR_SUMMARY_PATH.resolve()}')


## Red de tesis por asesor

Esta red conecta tesis asesoradas por la misma persona. El layout agrupa primero por cluster semántico para que las comunidades temáticas queden juntas, y mantiene las aristas de asesoría como puentes institucionales entre temas.


In [ ]:
graph_df = df_layout.copy()

G = nx.Graph()
G.add_nodes_from(graph_df.index)

for advisor, idxs in graph_df.groupby('asesor_unificado').groups.items():
    ids = list(idxs)
    if len(ids) < 2:
        continue
    for i, j in combinations(ids, 2):
        w = G.get_edge_data(i, j, {}).get('weight', 0) + 1
        G.add_edge(i, j, weight=w, advisor=advisor)

if G.number_of_edges() == 0:
    print('No hay conexiones por asesor (todos únicos o faltantes).')
    df_layout['advisor_comm_id'] = -1
    df_layout['advisor_comm_label'] = 'Sin comunidad'
else:
    communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight='weight'))
    comm_map = {node: cid for cid, nodes in enumerate(communities) for node in nodes}
    df_layout['advisor_comm_id'] = df_layout.index.map(lambda x: comm_map.get(x, -1))
    df_layout['advisor_comm_label'] = df_layout['advisor_comm_id'].map(lambda x: f'AdvisorComm {x}' if x != -1 else 'Sin comunidad')

    comm_summary = (
        df_layout.groupby('advisor_comm_label')
        .agg(
            conteo=('titulo', 'count'),
            asesores_top=('asesor_unificado', lambda s: ', '.join(s.value_counts().head(3).index)),
        )
        .sort_values('conteo', ascending=False)
    )
    print(f'Comunidades detectadas: {len(communities)}')
    display(comm_summary.head(20))


## Visualizar red de asesores agrupada por cluster semántico


In [ ]:
def hex_to_rgba(hex_color, alpha=0.12):
    hex_color = hex_color.lstrip('#')
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

def convex_hull(points):
    pts = sorted(set(map(tuple, points)))
    if len(pts) <= 2:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]

if 'advisor_comm_label' not in df_layout.columns or G.number_of_edges() == 0:
    fig_network = None
    print('No hay red de asesores para visualizar.')
else:
    cluster_counts = df_layout['cluster_label'].value_counts().reindex(cluster_order)
    n_clusters = len(cluster_order)
    n_cols = int(np.ceil(np.sqrt(n_clusters)))
    n_rows = int(np.ceil(n_clusters / n_cols))
    anchor_spacing_x = 4.7
    anchor_spacing_y = 3.8

    cluster_anchors = {}
    for order_idx, cluster_name in enumerate(cluster_order):
        row, col = divmod(order_idx, n_cols)
        x = (col - (n_cols - 1) / 2) * anchor_spacing_x
        y = ((n_rows - 1) / 2 - row) * anchor_spacing_y
        cluster_anchors[cluster_name] = np.array([x, y], dtype=float)

    pos = {}
    cluster_radii = {}
    for cid, cluster_name in enumerate(cluster_order):
        node_ids = df_layout.index[df_layout['cluster_label'].eq(cluster_name)].tolist()
        subgraph = G.subgraph(node_ids)
        radius = 0.72 + 0.06 * np.sqrt(len(node_ids))
        cluster_radii[cluster_name] = radius

        if len(node_ids) == 1:
            local_pos = {node_ids[0]: np.array([0.0, 0.0])}
        else:
            local_pos = nx.spring_layout(
                subgraph,
                weight='weight',
                seed=RANDOM_STATE + cid,
                k=0.9 / np.sqrt(max(len(node_ids), 2)),
                iterations=180,
            )
            coords = np.array([local_pos[node] for node in node_ids], dtype=float)
            coords = coords - coords.mean(axis=0)
            max_abs = np.abs(coords).max()
            if max_abs > 0:
                coords = coords / max_abs
            for node, coord in zip(node_ids, coords):
                local_pos[node] = coord

        anchor = cluster_anchors[cluster_name]
        for node in node_ids:
            degree_weight = G.degree(node, weight='weight') or 0
            pull_to_center = 1 - min(0.18, degree_weight * 0.012)
            pos[node] = anchor + np.asarray(local_pos[node]) * radius * pull_to_center

    nodes_df = (
        df_layout[['titulo', 'anio_pub', 'idioma', 'asesor_unificado', 'cluster_label', 'cluster_theme', 'cluster_display_label', 'advisor_comm_label']]
        .assign(
            x=lambda d: d.index.map(lambda i: pos[i][0]),
            y=lambda d: d.index.map(lambda i: pos[i][1]),
            degree=lambda d: d.index.map(lambda i: G.degree(i, weight='weight')),
        )
    )
    nodes_df['node_size'] = np.clip(6 + np.sqrt(nodes_df['degree'].fillna(0)) * 2.2, 6, 16)
    nodes_df['color'] = nodes_df['cluster_label'].map(cluster_color_map)

    hull_traces = []
    annotations = []
    for cluster_name in cluster_order:
        points = nodes_df.loc[nodes_df['cluster_label'].eq(cluster_name), ['x', 'y']].to_numpy()
        color = cluster_color_map.get(cluster_name, '#2563eb')
        anchor = cluster_anchors[cluster_name]

        if len(points) >= 3:
            hull = np.array(convex_hull(points))
            center = hull.mean(axis=0)
            hull = center + (hull - center) * 1.18
            hull = np.vstack([hull, hull[0]])
            hull_traces.append(
                go.Scatter(
                    x=hull[:, 0],
                    y=hull[:, 1],
                    mode='lines',
                    line=dict(color=hex_to_rgba(color, 0.42), width=1.1),
                    fill='toself',
                    fillcolor=hex_to_rgba(color, 0.10),
                    hoverinfo='skip',
                    showlegend=False,
                )
            )

        annotations.append(
            dict(
                x=anchor[0],
                y=anchor[1] + cluster_radii[cluster_name] + 0.45,
                text=f'{cluster_label_to_theme.get(cluster_name, cluster_name)}<br><span style="font-size:11px;color:#64748b">{cluster_name} · {int(cluster_counts.get(cluster_name, 0))} tesis</span>',
                showarrow=False,
                font=dict(size=12, color='#111827'),
                align='center',
                bgcolor='rgba(255,255,255,0.82)',
                bordercolor='rgba(148,163,184,0.45)',
                borderwidth=1,
                borderpad=4,
            )
        )

    intra_edge_x, intra_edge_y, cross_edge_x, cross_edge_y = [], [], [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        same_cluster = df_layout.loc[u, 'cluster_label'] == df_layout.loc[v, 'cluster_label']
        target_x = intra_edge_x if same_cluster else cross_edge_x
        target_y = intra_edge_y if same_cluster else cross_edge_y
        target_x += [x0, x1, None]
        target_y += [y0, y1, None]

    cross_edge_trace = go.Scatter(
        x=cross_edge_x,
        y=cross_edge_y,
        line=dict(width=0.55, color='rgba(15,23,42,0.16)', dash='dot'),
        hoverinfo='none',
        mode='lines',
        showlegend=False,
    )
    intra_edge_trace = go.Scatter(
        x=intra_edge_x,
        y=intra_edge_y,
        line=dict(width=0.50, color='rgba(71,85,105,0.18)'),
        hoverinfo='none',
        mode='lines',
        showlegend=False,
    )

    node_traces = []
    for cluster_name in cluster_order:
        subset = nodes_df[nodes_df['cluster_label'].eq(cluster_name)]
        if subset.empty:
            continue
        node_traces.append(
            go.Scatter(
                x=subset['x'],
                y=subset['y'],
                mode='markers',
                name=cluster_label_to_theme.get(cluster_name, cluster_name),
                marker=dict(
                    size=subset['node_size'],
                    color=cluster_color_map.get(cluster_name, '#2563eb'),
                    opacity=0.88,
                    line=dict(width=0.7, color='white'),
                ),
                text=subset.apply(
                    lambda r: (
                        f"{html.escape(str(r['titulo']))}<br>"
                        f"Asesor: {html.escape(str(r['asesor_unificado']))}<br>"
                        f"Tema: {html.escape(str(r['cluster_theme']))}<br>"
                        f"Cluster técnico: {html.escape(str(r['cluster_label']))}<br>"
                        f"Comunidad asesoría: {html.escape(str(r['advisor_comm_label']))}<br>"
                        f"Año: {html.escape(str(r['anio_pub']))}<br>"
                        f"Idioma: {html.escape(str(r['idioma']))}"
                    ),
                    axis=1,
                ),
                hoverinfo='text',
            )
        )

    fig_network = go.Figure(data=hull_traces + [cross_edge_trace, intra_edge_trace] + node_traces)
    fig_network.update_layout(
        title='Red de tesis por asesor agrupada por tema semántico',
        legend_title_text='Tema semántico',
        width=1080,
        height=780,
        margin=dict(l=25, r=25, t=72, b=25),
        plot_bgcolor='white',
        paper_bgcolor='white',
        annotations=annotations,
        xaxis=dict(visible=False, scaleanchor='y', scaleratio=1),
        yaxis=dict(visible=False),
    )
    fig_network.show()
    print('Layout de red: nodos agrupados por tema semántico; líneas sólidas dentro de tema y punteadas entre temas.')


## Dashboard HTML

Se exporta una vista interactiva en HTML con mapa temporal, diagnóstico de clusters, evolución temática, mezcla de idioma y cruces de asesoría.

In [ ]:

def fig_html(fig, div_id, include_js=False):
    if fig is None:
        return '<div class="empty-state">Sin datos suficientes para esta visualización.</div>'
    return pio.to_html(fig, include_plotlyjs=include_js, full_html=False, div_id=div_id)

def table_html(frame, columns, max_rows=12, formatters=None):
    present = [c for c in columns if c in frame.columns]
    if frame.empty or not present:
        return '<div class="empty-state">Sin filas para mostrar.</div>'
    view = frame[present].head(max_rows).copy()
    if formatters:
        for col, formatter in formatters.items():
            if col in view.columns:
                view[col] = view[col].map(formatter)
    return view.to_html(index=False, classes='data-table', border=0, escape=True)

selected_diag = diagnostics.query('feature_space == "embeddings" and k == @selected_k').iloc[0]
best_diag = diagnostics.query('feature_space == "embeddings"').sort_values(['silhouette', 'davies_bouldin'], ascending=[False, True]).iloc[0]
english_clusters = 0
if not cluster_language.empty and (cluster_language['idioma'] == 'eng').any():
    english_clusters = int(cluster_language.loc[cluster_language['idioma'] == 'eng', 'cluster_label'].nunique())

year_min = int(df_layout['anio_pub'].min())
year_max = int(df_layout['anio_pub'].max())
latest_year = int(df_layout['anio_pub'].max())
latest_total = int((df_layout['anio_pub'].astype('Int64') == latest_year).sum())
fastest_growth = cluster_lifecycle.sort_values('cambio_2020s_vs_2010s_pp', ascending=False).iloc[0]

metric_cards = [
    ('Tesis analizadas', f'{len(df_layout):,}'),
    ('Años cubiertos', f'{year_min}-{year_max}'),
    ('Clusters activos', f'{selected_k}'),
    ('Silhouette k usado', f'{selected_diag["silhouette"]:.3f}'),
    ('Trustworthiness UMAP', f'{umap_trust:.3f}'),
    ('Clusters con tesis en inglés', f'{english_clusters}'),
]
metric_html = ''.join(f'<section class="metric"><span>{html.escape(label)}</span><strong>{html.escape(value)}</strong></section>' for label, value in metric_cards)

cluster_table = table_html(cluster_overview.sort_values('cluster_id'), ['cluster_label', 'cluster_theme', 'conteo', 'anio_promedio', 'idiomas', 'keywords', 'asesores_top'], selected_k)
lifecycle_table = table_html(
    cluster_lifecycle.sort_values('cluster_id'),
    ['cluster_label', 'cluster_theme', 'conteo', 'anio_inicio', 'anio_pico', 'conteo_pico', 'participacion_pico', 'cambio_2020s_vs_2010s_pp', 'keywords'],
    selected_k,
    formatters={
        'participacion_pico': lambda x: f'{x:.1f}%',
        'cambio_2020s_vs_2010s_pp': lambda x: f'{x:+.1f} pp',
    },
)
advisor_table = table_html(advisor_summary, ['asesor_unificado', 'n_tesis', 'n_clusters', 'cluster_principal', 'cluster_principal_id', 'participacion_cluster_principal', 'anio_min', 'anio_max'], 18)
benchmark_table = table_html(benchmark_view if 'benchmark_view' in globals() else pd.DataFrame(), ['model_name', 'embedding_dim', 'seconds', 'k', 'silhouette_cosine', 'language_nmi', 'umap_trustworthiness'], 10)

time_machine_html = fig_html(fig_time_machine, 'time-machine', include_js=True)
periods_html = fig_html(fig_period_facets, 'period-facets')
map_html = fig_html(fig_map, 'semantic-map')
diag_html = fig_html(fig_diag, 'cluster-diagnostics')
year_html = fig_html(fig_year_heatmap, 'year-heatmap')
stream_html = fig_html(fig_theme_stream, 'theme-stream')
race_html = fig_html(fig_theme_race, 'theme-race')
language_html = fig_html(fig_language, 'language-distribution')
advisor_html = fig_html(fig_advisor, 'advisor-scatter')
network_html = fig_html(fig_network, 'advisor-network')

html_doc = f'''<!doctype html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Mapa semántico de tesis CIDE</title>
<style>
:root {{ --ink:#111827; --muted:#64748b; --line:#d8dee8; --panel:#fff; --soft:#f6f8fb; --accent:#2563eb; --accent2:#d97706; }}
* {{ box-sizing:border-box; }}
html {{ scroll-behavior:smooth; }}
body {{ margin:0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; color:var(--ink); background:#f8fafc; }}
header {{ padding:28px 32px 18px; border-bottom:1px solid var(--line); background:#fff; }}
h1 {{ margin:0 0 8px; font-size:30px; line-height:1.15; letter-spacing:0; }}
header p {{ margin:0; color:var(--muted); max-width:1120px; line-height:1.45; }}
nav {{ display:flex; gap:8px; flex-wrap:wrap; margin-top:16px; }}
nav a {{ color:#1f2937; text-decoration:none; border:1px solid var(--line); background:#fff; border-radius:8px; padding:7px 10px; font-size:13px; }}
main {{ padding:24px 32px 40px; }}
.metrics {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:12px; margin-bottom:18px; }}
.metric {{ background:var(--panel); border:1px solid var(--line); border-radius:8px; padding:13px 14px; }}
.metric span {{ display:block; color:var(--muted); font-size:12px; }}
.metric strong {{ display:block; margin-top:4px; font-size:24px; }}
.grid {{ display:grid; grid-template-columns:minmax(0,1fr); gap:18px; }}
.panel {{ background:var(--panel); border:1px solid var(--line); border-radius:8px; padding:16px; overflow:hidden; }}
.panel h2 {{ margin:0 0 10px; font-size:18px; }}
.panel p {{ color:var(--muted); line-height:1.45; }}
.two-col {{ display:grid; grid-template-columns:minmax(0,1fr) minmax(0,1fr); gap:18px; }}
.callout {{ border-left:4px solid var(--accent); padding:8px 12px; background:#f8fafc; margin:0 0 12px; color:#243043; }}
.data-table {{ width:100%; border-collapse:collapse; font-size:13px; }}
.data-table th,.data-table td {{ border-bottom:1px solid var(--line); padding:8px 10px; vertical-align:top; text-align:left; }}
.data-table th {{ background:var(--soft); font-weight:650; }}
.data-table td {{ color:#243043; }}
.empty-state {{ padding:24px; color:var(--muted); background:var(--soft); border-radius:8px; }}
.note {{ margin-top:8px; color:var(--muted); font-size:13px; }}
@media (max-width:900px) {{ header,main {{ padding-left:16px; padding-right:16px; }} .two-col {{ grid-template-columns:1fr; }} h1 {{ font-size:24px; }} nav a {{ padding:7px 9px; }} }}
</style></head><body>
<header><h1>Mapa semántico de tesis CIDE</h1><p>Embeddings multilingües con {html.escape(MODEL_NAME)}, UMAP como layout visual y K-Means sobre embeddings con k={selected_k}. Generado el {html.escape(datetime.now().strftime('%Y-%m-%d %H:%M'))}.</p><nav><a href="#tiempo">Tiempo</a><a href="#mapa">Mapa</a><a href="#clusters">Clusters</a><a href="#asesores">Asesores</a><a href="#metodo">Método</a></nav></header>
<main><section class="metrics">{metric_html}</section><section class="grid">
<article class="panel" id="tiempo"><h2>Máquina temporal del mapa semántico</h2><p class="callout">El periodo más reciente tiene {latest_total} tesis en {latest_year}. El tema con mayor crecimiento relativo frente a 2010s es {html.escape(str(fastest_growth['cluster_theme']))} ({fastest_growth['cambio_2020s_vs_2010s_pp']:+.1f} pp).</p>{time_machine_html}</article>
<section class="two-col"><article class="panel"><h2>Mapa por periodos</h2>{periods_html}</article><article class="panel"><h2>Carrera anual de temas</h2>{race_html}</article></section>
<article class="panel"><h2>Composición temática anual</h2>{stream_html}</article>
<article class="panel"><h2>Ciclo de vida de los temas</h2>{lifecycle_table}</article>
<article class="panel" id="mapa"><h2>Mapa semántico acumulado</h2>{map_html}</article>
<section class="two-col"><article class="panel"><h2>Diagnóstico de clusters</h2>{diag_html}<p class="note">Mejor k por silhouette: {int(best_diag['k'])}; k usado: {selected_k} para mayor granularidad interpretativa.</p></article><article class="panel" id="metodo"><h2>Benchmark de modelos</h2>{benchmark_table}</article></section>
<article class="panel" id="clusters"><h2>Resumen por cluster</h2>{cluster_table}</article>
<section class="two-col"><article class="panel"><h2>Heatmap año × tema</h2>{year_html}</article><article class="panel"><h2>Distribución por idioma y tema</h2>{language_html}</article></section>
<section class="two-col" id="asesores"><article class="panel"><h2>Asesores: volumen y diversidad</h2>{advisor_html}</article><article class="panel"><h2>Top asesores</h2>{advisor_table}</article></section>
<article class="panel"><h2>Red de tesis por asesor y tema</h2>{network_html}</article>
</section></main></body></html>'''

DASHBOARD_PATH.write_text(html_doc, encoding='utf-8')
print(f'Dashboard HTML exportado: {DASHBOARD_PATH.resolve()}')
